In [ ]:
#  Copyright 2023 Google LLC
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

# Unary Iteration

Given an array of potential operations, for example:

    ops = [X(i) for i in range(5)]
    
we would like to select an operation to apply:

    n = 4 --> apply ops[4]
    
If $n$ is a quantum integer, we need to apply the transformation

$$
    |n \rangle |\psi\rangle \rightarrow |n\rangle \, \mathrm{ops}_n \cdot |\psi\rangle
$$

The simplest conceptual way to do this is to use a "total control" quantum circuit where you introduce a multi-controlled operation for each of the `len(ops)` possible `n` values.

In [ ]:
from itertools import product
from collections import Counter
import torch
from sympy import And, Not, true, symbols
from examples.algorithms.unary_iteration_catalog import UnaryIntervalExample, UnaryXExample
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.compiler.registers.control.multi_control import multi_control_x_ir
from flagquantum.core.ir import CircuitIR
from flagquantum.drawer.text_drawer import draw_text
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

torch.set_num_threads(1)

In [ ]:
def binary_digits(value, width):
    return tuple((value >> shift) & 1 for shift in reversed(range(width)))

## Total Control

Here, we'll use Sympy's boolean logic to show how total control works. We perform an `And( ... )` for each possible bit pattern. We use an `Xnor` on each selection bit to toggle whether it's a positive or negative control (filled or open circle in quantum circuit diagrams).

In this example, we indeed consider $X_n$ as our potential operations and toggle bits in the `target` register according to the total control.

In [ ]:
def total_control(selection):
    return tuple(
        And(*(bit if active else Not(bit) for bit, active in zip(selection, pattern)))
        for pattern in product((0, 1), repeat=len(selection))
    )

print(total_control((False, True, False)))

Note that our target register shows we have indeed applied $X_\mathrm{0b010}$. Try changing `selection` to other bit patterns and notice how it changes.

Of course, we don't know what state the selection register will be in. We can use sympy's support for symbolic boolean logic to verify our gadget for all possible selection inputs.

In [ ]:
selector = symbols("s:3")
direct = total_control(selector)
for address, predicate in enumerate(direct):
    print(address, predicate)

As expected, the "not pattern" (where `~` is boolean not) matches the binary representations of `n`.

## Unary Iteration with segment trees

A [segment tree](https://en.wikipedia.org/wiki/Segment_tree) is a data structure that allows logrithmic-time querying of intervals. We use a segment tree where each interval is length 1 and comprises all the `n` integers we may select.

It is defined recursively by dividing the input interval into two half-size intervals until the left limit meets the right limit.

In [ ]:
def segment_tree(selection, prefix=true):
    if not selection:
        return (prefix,)
    bit, *rest = selection
    return (*segment_tree(rest, And(prefix, Not(bit))),
            *segment_tree(rest, And(prefix, bit)))

In [ ]:
tree = segment_tree(selector)
for address, predicate in enumerate(tree):
    print(address, predicate)

In [ ]:
assert tree == direct
print("Both constructions select the same eight rows.")

## Quantum Circuit

We can translate the boolean logic to reversible, quantum logic. It is instructive to start from the suboptimal total control quantum circuit for comparison purposes. We can build this as in the sympy boolean-logic case by adding controlled X operations to the target signature, with the controls on the selection signature toggled on or off according to the binary representation of the selection index.

Let us first build a GateWithRegisters object to implement the circuit

In [ ]:
def fully_controlled_x():
    control_and_selection = (0, 1, 2, 3)
    workspace = (9, 10, 11)
    return tuple(
        gate
        for address in range(5)
        for gate in multi_control_x_ir(
            control_and_selection, (1, *binary_digits(address, 3)), 8 - address,
            workspace, (0, 1, 2)
        )
    )

def assert_selected_target(state_width, instructions, address, start, n_targets, work):
    prefix = ((1 << (start - 1)) | address) << (n_targets + len(work))
    initial = torch.zeros(1 << state_width, dtype=torch.complex128)
    initial[prefix] = 1
    expected = torch.zeros_like(initial)
    expected[prefix | (1 << (address + len(work)))] = 1
    circuit = DynamicCircuit(state_width, inputs=initial, dtype=torch.complex128)
    circuit.compose(CircuitIR(state_width, instructions))
    result = run_dynamic(circuit, shots=2, seed=7, strategy="trajectory")
    torch.testing.assert_close(result.final_states, expected.expand(2, -1), rtol=0, atol=2e-12)

In [ ]:
total_stream = fully_controlled_x()
total_ir = CircuitIR(12, total_stream)
counts = Counter(op.name for op in total_ir.instructions)
print("Fully controlled X:", counts)
# The first fully controlled X, including its clean-work uncomputation.
print(draw_text(CircuitIR(12, multi_control_x_ir((0, 1, 2, 3), (1, 0, 0, 0),
                                                8, (9, 10, 11), (0, 1, 2)))))

## Tests for Correctness

We can use a full statevector simulation to compare the desired statevector to the one generated by the unary iteration circuit for each basis state.

In [ ]:
for address in range(5):
    assert_selected_target(12, total_stream, address, 4, 5, (9, 10, 11))
print("All five independently controlled targets checked.")

## Towards a segment tree 

Next let's see how we can reduce the circuit to the observe the tree structure.
First let's recall what we are trying to do with the controlled not. Given a
selection integer (say 3 = 011), we want to toggle the bit in the target
register to on if the qubit 1 and 2 are set to on in the selection register.

In [ ]:
assert_selected_target(12, total_stream, 3, 4, 5, (9, 10, 11))
print("Address 011 sets the fourth target from the right.")

Now what is important to note is that we can remove many repeated controlled operations by using ancilla qubits to flag what part of the circuit we need to apply, this works because we know the bit pattern of nearby integers is very similar. 

A circuit demonstrating this for our example is given below.

In [ ]:
small = UnaryXExample(selection_width=2, targets=4)
circuit = DynamicCircuit(small.qubit_count)
circuit.append(small)
print(draw_text(circuit.to_ir()))

Reading from left to right we first check the control is set to on and the selection qubit is off, if both these conditions are met then the ancilla qubit is now set to 1. The next control checks if the previous condition was met and likewise the second selection index is also off. At this point if both these conditions are met we must be indexing 0 as the first two qubits are set to off (00), otherwise we know that we want to apply X to qubit 1 so we perform a CNOT operation to flip the bit value in the second ancilla qubit, before returning back up the circuit. Now if the left half of the circuit was not applied (i.e. the first selection register was set to 1) then the CNOT between the control qubit and the first ancilla qubit causes the ancilla qubit to toggle on. This triggers the right side of the circuit, which now performs the previously described operations to figure out if the lowest bit is set. Combining these two then yields the expected controlled X operation. 

Below we check the circuit is giving the expected behaviour.

In [ ]:
assert_selected_target(small.qubit_count, circuit.to_ir().instructions,
                       3, 3, 4, (7, 8))
print("The shared predicate traversal sets the same target.")

Extending the above idea to larger ranges of integers is relatively straightforward. For example consider the next simplest case of $L=8 = 2^3$. The circuit above takes care of the last two bits and can be duplicated. For the extra bit we just need to add a additional `AND` operations, and a CNOT to switch between the original range `[0,3]` or the new range `[4,7]` depending on whether the new selection register is off or on respectively. This procedure can be repeated and we can begin to notice the recursive tree-like structure. 

This structure is just the segtree described previously for boolean logic and this gives is the basic idea of unary iteration, 
which uses `L-1` `AND` operations. Below the `ApplyXToLthQubit` builds the controlled Not operation using the `UnaryIterationGate` as a base class which defines the `decompose_from_registers` method appropriately to recursively construct the unary iteration circuit.

Note below a different ordering of ancilla and selection qubits is taken to what was used in the simpler `L=4` example.

In [ ]:
bounded = UnaryXExample()
loop = UnaryIntervalExample()

In [ ]:
circuit = DynamicCircuit(bounded.qubit_count)
circuit.append(bounded)
bounded_ir = circuit.to_ir()
counts = Counter(op.name for op in bounded_ir.instructions)
print("Emitted gates:", counts)
print("Declared model:", estimate_component_resources(bounded.resource_model()))
interval = DynamicCircuit(loop.qubit_count)
interval.append(loop)
print("Nonzero interval cost:", estimate_component_resources(loop.resource_model()))

## Tests for Correctness

We can use a full statevector simulation to check again that the optimized circuit produces the expected result.

In [ ]:
for address in range(5):
    assert_selected_target(bounded.qubit_count, bounded_ir.instructions, address, 4, 5, (9, 10, 11))
print("All five bounded targets checked.")